# 06. Machine Learning Models

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from lightgbm import LGBMRegressor
from sklearn.compose import TransformedTargetRegressor

# Add project root to path
import os
import sys
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_data
from src.splitting import split_train_test
from src.evaluation import evaluate, highlight_table
from src.models.baseline import seasonal_naive_forecast
from src.models.ml import *
from src.models.optimizations import *

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Python version: 3.10.11
Pandas version: 2.1.4
NumPy version: 1.26.4


In [2]:
# Load data using custom loader
try:
    df = load_data('data/processed/data_total_features.parquet')
    print("Data loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Time span: {df.index.min()} → {df.index.max()}")
    if not df.index.freq:
        df = df.asfreq("1H")
    print(f"Frequency: {df.index.freq}")
except Exception as e:
    print(f"[Error] {e}")

Loading: data/processed/data_total_features.parquet
Data loaded successfully!
Shape: (35064, 11)
Time span: 2011-01-01 00:00:00 → 2014-12-31 23:00:00
Frequency: <Hour>


In [3]:
df_split = df.copy()
df_split = df_split[df.index >= "2012-01-01"]
df_split.head()

,total_load,hour,month,day_of_week,day_of_month,day_of_year,hour_sin,hour_cos,year_sin,year_cos,holiday_name
Timestamp,,,,,,,,,,,
2012-01-01 00:00:00,104230.493821,0,1,6,1,1,0.000000,1.000000,0.0,1.0,Ano Novo
2012-01-01 01:00:00,112550.368868,1,1,6,1,1,0.258819,0.965926,0.0,1.0,Ano Novo
2012-01-01 02:00:00,110930.247627,2,1,6,1,1,0.500000,0.866025,0.0,1.0,Ano Novo
2012-01-01 03:00:00,106191.754971,3,1,6,1,1,0.707107,0.707107,0.0,1.0,Ano Novo
2012-01-01 04:00:00,103540.419166,4,1,6,1,1,0.866025,0.500000,0.0,1.0,Ano Novo


In [4]:
X_train, X_test, y_train, y_test, _ = split_train_test(df_split, 'total_load', split_time="2014-01-01", categorical_cols='holiday_name')

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (17568, 10)
Test shape: (8759, 10)


## 1. Baseline

In [5]:
# Naive forecast
seasonal_naive_pred = seasonal_naive_forecast(df_split, y_test, season=365)

# Evaluate forecasts
seasonal_naive_metrics = evaluate(y_test, seasonal_naive_pred)

print("Seasonal Naive (1 year):", seasonal_naive_metrics)

Seasonal Naive (1 year): {'MAE': 14598.19, 'RMSE': 21626.9, 'MAPE': 6.07}


## 2. Models

### 2.1 Random Forest

In [6]:
rdf, rdf_pred = random_forest_model(X_train, y_train, X_test)
rdf_metrics = evaluate(y_test, rdf_pred)

print("Random Forest:", rdf_metrics)

Random Forest: {'MAE': 13265.68, 'RMSE': 19425.27, 'MAPE': 5.56}


### 2.2 LightGBM

In [7]:
lgbm, lgbm_pred = lightgbm_model(X_train, y_train, X_test)
lgbm_metrics = evaluate(y_test, lgbm_pred)

print("LightGBM:", lgbm_metrics)

LightGBM: {'MAE': 13629.68, 'RMSE': 19854.71, 'MAPE': 5.8}


### 2.3 XGBoost

In [8]:
xgb, xgb_pred = xgboost_model(X_train, y_train, X_test)
xgb_metrics = evaluate(y_test, xgb_pred)

print("XGBoost:", xgb_metrics)

XGBoost: {'MAE': 13554.65, 'RMSE': 20252.6, 'MAPE': 5.62}


## 3. Revive 2011

In [9]:
# Đảm bảo df_clean gốc được sắp xếp chuẩn theo DatetimeIndex
df_clean = df.sort_index()
target = 'total_load'

print("--- BẮT ĐẦU QUY TRÌNH: TẢI API THỜI TIẾT & PHỤC DỰNG 2011 ---")

# =========================================================================
# BƯỚC 1: TẢI DỮ LIỆU THỜI TIẾT TỪ OPEN-METEO API (MIỄN PHÍ)
# =========================================================================
# Sử dụng tọa độ Lisbon, Bồ Đào Nha (Lat: 38.7167, Lon: -9.1333) làm proxy
# Chỉ cần tải từ 2011-01-01 đến 2013-12-31 (phục vụ riêng cho Backcasting)
print("1. Đang kết nối API Open-Meteo tải dữ liệu thời tiết quá khứ...")

url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    "latitude=38.7167&longitude=-9.1333"
    "&start_date=2011-01-01&end_date=2013-12-31"
    "&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    "&timezone=Europe%2FLisbon"
)

response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Lỗi tải API: {response.text}")

data = response.json()
hourly = data['hourly']

# Chuyển đổi JSON thành DataFrame
df_weather_api = pd.DataFrame({
    'time': pd.to_datetime(hourly['time']),
    'temp': hourly['temperature_2m'],
    'humidity': hourly['relative_humidity_2m'],
    'wind_speed': hourly['wind_speed_10m']
}).set_index('time')

# Loại bỏ thông tin múi giờ (tz-naive) để an toàn gộp với df_clean của bạn
if df_weather_api.index.tz is not None:
    df_weather_api.index = df_weather_api.index.tz_localize(None)

print(f"-> Tải thành công {len(df_weather_api)} dòng dữ liệu thời tiết!")


# =========================================================================
# BƯỚC 2: GỘP DỮ LIỆU & FEATURE ENGINEERING THỜI TIẾT
# =========================================================================
print("2. Gộp dữ liệu và trích xuất các đặc trưng khí hậu bậc cao...")

# Gộp (Join) thời tiết vào bản sao của df_clean dựa trên khớp mốc giờ
df_process = df_clean.loc['2011-01-01':'2013-12-31'].copy()
df_process = df_process.join(df_weather_api, how='left')

# Xử lý nội suy nếu API có sót vài giờ ngẫu nhiên
df_process[['temp', 'humidity', 'wind_speed']] = df_process[['temp', 'humidity', 'wind_speed']].interpolate(method='time')

# --- Tạo Features Khí hậu ---
# 2.1. CDD/HDD (Làm mát > 18°C, Sưởi ấm < 15°C)
df_process['CDD'] = np.maximum(0, df_process['temp'] - 18)
df_process['HDD'] = np.maximum(0, 15 - df_process['temp'])
df_process['CDD_squared'] = df_process['CDD'] ** 2

# 2.2. Quán tính nhiệt (Ngậm nhiệt sau 3 ngày nắng gắt)
df_process['temp_rolling_mean_72h'] = df_process['temp'].rolling(window=72).mean()

# 2.3. Chỉ số oi bức (Apparent Temperature / THI)
T = df_process['temp']
RH = df_process['humidity']
df_process['THI_Apparent'] = T - (0.55 - 0.0055 * RH) * (T - 14.5)

# 2.4. Gió làm mát khi trời đang nóng
df_process['wind_cooling_effect'] = df_process['wind_speed'] * df_process['CDD']

# 2.5. Tương tác hành vi: Giờ x Nhiệt độ
if 'hour' in df_process.columns:
    df_process['hour_x_CDD'] = df_process['hour'] * df_process['CDD']

# Lấp đầy mốc NaN do rolling ở các dòng đầu
df_process = df_process.bfill()


# =========================================================================
# BƯỚC 3: HUẤN LUYỆN "CỖ MÁY THỜI GIAN" VỚI LIGHTGBM
# =========================================================================
# Lấy toàn bộ features ngoại sinh gốc + features thời tiết vừa tạo
exo_features = [col for col in df_process.columns if col != target]

# Tập Train: Giai đoạn trưởng thành (2012 và 2013)
X_train_stable = df_process.loc['2012-01-01':'2013-12-31', exo_features]
y_train_stable = df_process.loc['2012-01-01':'2013-12-31', target]

# Tập Cần phục dựng (Hindcast): Toàn bộ năm 2011
X_backcast_2011 = df_process.loc['2011-01-01':'2011-12-31', exo_features]

lgbm_core = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

backcast_model = TransformedTargetRegressor(
    regressor=lgbm_core,
    func=np.log1p,
    inverse_func=np.expm1
)

print("3. Đang huấn luyện mô hình học quy luật tải theo thời tiết từ 2012-2013...")
backcast_model.fit(X_train_stable, y_train_stable)


# =========================================================================
# BƯỚC 4: TÁI TẠO 2011 & TRẢ VỀ KẾT QUẢ SẠCH (BỎ THỜI TIẾT)
# =========================================================================
print("4. Đang ánh xạ ngược để tái tạo sản lượng điện năm 2011...")
y_synthetic_2011 = backcast_model.predict(X_backcast_2011)

# Lấy lại tập df_clean GỐC của bạn (để đảm bảo không dính dáng gì đến cột thời tiết)
df_2011 = df.copy()

# THAY THẾ HOÀN TOÀN tải điện thô 2011 bằng chuỗi tổng hợp chuẩn thời tiết
df_2011.loc['2011-01-01':'2011-12-31', target] = y_synthetic_2011

print("--- HOÀN TẤT QUY TRÌNH! ---")
print("-> Toàn bộ dữ liệu thời tiết tải từ API đã làm xong nhiệm vụ đòn bẩy và ĐÃ BỊ LOẠI BỎ.")
print("-> Tập df_2011 hiện tại giữ nguyên cấu trúc cột ban đầu của bạn.")
print("-> Đỉnh tải điện 2011 đã được nâng khớp hoàn hảo. Bạn có thể tạo lag_365 ngay!")

df_2011.head()

--- BẮT ĐẦU QUY TRÌNH: TẢI API THỜI TIẾT & PHỤC DỰNG 2011 ---
1. Đang kết nối API Open-Meteo tải dữ liệu thời tiết quá khứ...
-> Tải thành công 26304 dòng dữ liệu thời tiết!
2. Gộp dữ liệu và trích xuất các đặc trưng khí hậu bậc cao...
3. Đang huấn luyện mô hình học quy luật tải theo thời tiết từ 2012-2013...
4. Đang ánh xạ ngược để tái tạo sản lượng điện năm 2011...
--- HOÀN TẤT QUY TRÌNH! ---
-> Toàn bộ dữ liệu thời tiết tải từ API đã làm xong nhiệm vụ đòn bẩy và ĐÃ BỊ LOẠI BỎ.
-> Tập df_2011 hiện tại giữ nguyên cấu trúc cột ban đầu của bạn.
-> Đỉnh tải điện 2011 đã được nâng khớp hoàn hảo. Bạn có thể tạo lag_365 ngay!


,total_load,hour,month,day_of_week,day_of_month,day_of_year,hour_sin,hour_cos,year_sin,year_cos,holiday_name
Timestamp,,,,,,,,,,,
2011-01-01 00:00:00,128273.479673,0,1,5,1,1,0.000000,1.000000,0.0,1.0,Ano Novo
2011-01-01 01:00:00,109173.297911,1,1,5,1,1,0.258819,0.965926,0.0,1.0,Ano Novo
2011-01-01 02:00:00,105270.467688,2,1,5,1,1,0.500000,0.866025,0.0,1.0,Ano Novo
2011-01-01 03:00:00,103119.465761,3,1,5,1,1,0.707107,0.707107,0.0,1.0,Ano Novo
2011-01-01 04:00:00,100037.338338,4,1,5,1,1,0.866025,0.500000,0.0,1.0,Ano Novo


In [10]:
X_train_backcast, X_test_backcast, y_train_backcast, y_test_backcast, _ = split_train_test(df_2011, 'total_load', split_time="2014-01-01", categorical_cols='holiday_name')

print("Train shape:", X_train_backcast.shape)
print("Test shape:", X_test_backcast.shape)

Train shape: (26328, 10)
Test shape: (8759, 10)


In [11]:
rdf_backcast, rdf_backcast_pred = random_forest_model(X_train_backcast, y_train_backcast, X_test_backcast)
rdf_backcast_metrics = evaluate(y_test_backcast, rdf_backcast_pred)

print("Random Forest:", rdf_backcast_metrics)

Random Forest: {'MAE': 12828.45, 'RMSE': 18776.22, 'MAPE': 5.36}


In [12]:
lgbm_backcast, lgbm_backcast_pred = lightgbm_model(X_train_backcast, y_train_backcast, X_test_backcast)
lgbm_backcast_metrics = evaluate(y_test_backcast, lgbm_backcast_pred)

print("LightGBM:", lgbm_backcast_metrics)

LightGBM: {'MAE': 13012.48, 'RMSE': 18869.57, 'MAPE': 5.58}


In [13]:
xgb_backcast, xgb_backcast_pred = xgboost_model(X_train_backcast, y_train_backcast, X_test_backcast)
xgb_backcast_metrics = evaluate(y_test_backcast, xgb_backcast_pred)

print("XGBoost:", xgb_backcast_metrics)

XGBoost: {'MAE': 12956.09, 'RMSE': 19420.27, 'MAPE': 5.37}


## 4. Optimization

In [14]:
df_optimize = df_2011.copy()
df_optimize = df_optimize[df.index < '2014-01-01']
df_optimize.head()

,total_load,hour,month,day_of_week,day_of_month,day_of_year,hour_sin,hour_cos,year_sin,year_cos,holiday_name
Timestamp,,,,,,,,,,,
2011-01-01 00:00:00,128273.479673,0,1,5,1,1,0.000000,1.000000,0.0,1.0,Ano Novo
2011-01-01 01:00:00,109173.297911,1,1,5,1,1,0.258819,0.965926,0.0,1.0,Ano Novo
2011-01-01 02:00:00,105270.467688,2,1,5,1,1,0.500000,0.866025,0.0,1.0,Ano Novo
2011-01-01 03:00:00,103119.465761,3,1,5,1,1,0.707107,0.707107,0.0,1.0,Ano Novo
2011-01-01 04:00:00,100037.338338,4,1,5,1,1,0.866025,0.500000,0.0,1.0,Ano Novo


In [15]:
X_optuna, X_valid, y_optuna, y_valid, _ = split_train_test(df_optimize, 'total_load', split_time="2013-01-01", categorical_cols='holiday_name')

print("Train shape:", X_optuna.shape)
print("Val shape:", X_valid.shape)

Train shape: (17568, 10)
Val shape: (8759, 10)


In [16]:
rdf_best = optuna_random_forest(X_optuna, y_optuna, X_valid, y_valid, n_trials=50)

MODEL      : random_forest
BEST RMSE  : 20283.7575
BEST PARAMS:
{'n_estimators': 914, 'max_depth': 10, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 0.8368078049885144}


In [17]:
lgbm_best = optuna_lightgbm(X_optuna, y_optuna, X_valid, y_valid, n_trials=50)

MODEL      : lightgbm
BEST RMSE  : 20020.4306
BEST PARAMS:
{'n_estimators': 937, 'learning_rate': 0.007791311812192606, 'max_depth': 4, 'num_leaves': 176, 'subsample': 0.9954627320949059, 'colsample_bytree': 0.9993134344664012, 'min_child_samples': 30, 'reg_alpha': 0.002015220146534741, 'reg_lambda': 7.285146928910762}


In [18]:
xgb_best = optuna_xgboost(X_optuna, y_optuna, X_valid, y_valid, n_trials=50)

MODEL      : xgboost
BEST RMSE  : 20002.3028
BEST PARAMS:
{'n_estimators': 940, 'learning_rate': 0.007803666469435044, 'max_depth': 4, 'subsample': 0.8745355472945856, 'colsample_bytree': 0.9956789026108818, 'min_child_weight': 18, 'gamma': 2.4860413952052443, 'reg_alpha': 0.005411220270206435, 'reg_lambda': 9.527102811372876}


In [19]:
X_train_optimize, X_test_optimize, y_train_optimize, y_test_optimize, _ = split_train_test(df_2011, 'total_load', split_time="2014-01-01", categorical_cols='holiday_name')

print("Train shape:", X_train_optimize.shape)
print("Test shape:", X_test_optimize.shape)

Train shape: (26328, 10)
Test shape: (8759, 10)


In [20]:
rdf_optimize, rdf_optimize_pred = random_forest_model(X_train_optimize, y_train_optimize, X_test_optimize, rdf_best.best_params)
rdf_optimize_metrics = evaluate(y_test_optimize, rdf_optimize_pred)

print("Random Forest:", rdf_optimize_metrics)

Random Forest: {'MAE': 13093.43, 'RMSE': 18439.95, 'MAPE': 5.75}


In [21]:
lgbm_optimize, lgbm_optimize_pred = lightgbm_model(X_train_optimize, y_train_optimize, X_test_optimize, lgbm_best.best_params)
lgbm_optimize_metrics = evaluate(y_test_optimize, lgbm_optimize_pred)

print("LightGBM:", lgbm_optimize_metrics)

LightGBM: {'MAE': 13063.47, 'RMSE': 18310.15, 'MAPE': 5.71}


In [22]:
xgb_optimize, xgb_optimize_pred = xgboost_model(X_train_optimize, y_train_optimize, X_test_optimize, xgb_best.best_params)
xgb_optimize_metrics = evaluate(y_test_optimize, xgb_optimize_pred)

print("XGBoost:", xgb_optimize_metrics)

XGBoost: {'MAE': 13947.85, 'RMSE': 19772.9, 'MAPE': 5.94}


## 5. Compare


Below is a summary table of model performance across three phases: Before 2011 data, After 2011 reconstruction (adding features), and After hyperparameter tuning.

In [30]:
baseline_metrics = {"Seasonal Naive (1 year)": seasonal_naive_metrics}
without_2011_metrics = {
    "Random Forest": rdf_metrics,
    "LightGBM": lgbm_metrics,
    "XGBoost": xgb_metrics
}
backcast_metrics = {
    "Random Forest": rdf_backcast_metrics, 
    "LightGBM": lgbm_backcast_metrics,
    "XGBoost": xgb_backcast_metrics
}
optimized_metrics = {
    "Random Forest": rdf_optimize_metrics,
    "LightGBM": lgbm_optimize_metrics,
    "XGBoost": xgb_optimize_metrics
}
all_metrics = {
    "Baseline": baseline_metrics,
    "Without 2011": without_2011_metrics,
    "Backcast": backcast_metrics,
    "Optimized": optimized_metrics
}

rows = []
for group_name, group in all_metrics.items():
    for model_name, metrics in group.items():
        rows.append({
            "Group": group_name,
            "Model": model_name,
            "MAE": round(metrics["MAE"], 2),
            "RMSE": round(metrics["RMSE"], 2),
            "MAPE (%)": round(metrics["MAPE"], 2)
        })

comparison_df = pd.DataFrame(rows).set_index(["Group", "Model"])
comparison_df.style.apply(highlight_table).format("{:.2f}")